Bootstrap & install instructions

In [ ]:
# Bootstrap and installation

!pip install --quiet anthropic pydantic astra_swarm
!pip install --upgrade --quiet ipython
from google.colab import userdata, drive
drive.mount('/content/drive')

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")
print('Ready!')

Generate realistic auth logs

In [ ]:
from astra_swarm.alerts import _ask_structured
from astra_swarm.cassette import cassette
from pydantic import BaseModel, Field
from typing import Literal
import json
from pathlib import Path


class AuthLogEntry(BaseModel):
    ts: str = Field(description="ISO 8601 UTC timestamp")
    user: str
    src_ip: str
    geo: str = Field(description="City, Country")
    result: Literal["success", "fail"]
    mfa_challenge: bool
    device_id: str
    reason: str = Field(description="Short reason if result is fail, empty otherwise")


class AuthLogFixture(BaseModel):
    entries: list[AuthLogEntry]


with cassette("day_a_authlogs_fixture"):
    prompt = """Generate 100 realistic auth log entries covering ~10 distinct users over
the last 48 hours. Include:
- Baseline benign activity for most users
- One user with clear impossible-travel (logins from 2 continents within 10 minutes)
- One user with MFA-fatigue pattern (30+ MFA challenges in 15 minutes, all failed)
- One user with dormant account suddenly reactivating from a new geo
- One user with credential-spray failures (many users, one IP, similar timestamps)
Timestamps in ISO 8601 UTC. Realistic emails at example.com."""

    fixture = _ask_structured(prompt, AuthLogFixture, max_tokens=6000)

out = Path("/content/astra-swarm/data/synthetic/auth_logs.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps([e.model_dump() for e in fixture.entries], indent=2))
print(f"wrote {len(fixture.entries)} entries")